Based on:

- https://github.com/csheneka/ML-for-Astro-tutorial/blob/main/spectral_classifier.ipynb
- https://ogrisel.github.io/scikit-learn.org/sklearn-tutorial/tutorial/astronomy/

In [ ]:
import time
import cupy as cp
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import joblib
import scipy.stats as stats
import pandas as pd

from cuml.preprocessing import Normalizer
from cuml.decomposition import PCA
from cuml.linear_model import LogisticRegression
from cuml.ensemble import RandomForestClassifier

from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    StratifiedKFold
)

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

from sklearn.base import BaseEstimator, ClassifierMixin

In [ ]:
data_path = "../data/spectra/"

# ============================================================
# LOAD TO GPU
# ============================================================

data = cp.asarray(np.load(data_path + "data.npy"))
labels = cp.asarray(np.load(data_path + "labels.npy"))
wavelengths = cp.asarray(np.load(data_path + "wavelengths.npy"))

class_names = ["AGN", "galaxy", "QSO"]

print("Data shape:", data.shape)
print("Labels shape:", labels.shape)
print("Wavelength bins:", wavelengths.shape)

unique, counts = cp.unique(labels, return_counts=True)

for u, c in zip(cp.asnumpy(unique), cp.asnumpy(counts)):
    print(f"Class {class_names[u]}: {c} samples")

In [ ]:
f, axs = plt.subplots(1, 3, figsize=(12,3))

rng = np.random.default_rng(12345)

for i, cls in enumerate(cp.unique(labels).get()):

    idx = cp.where(labels == cls)[0].get()

    n = rng.choice(idx)

    axs[i].plot(
        wavelengths.get(),
        data[n].get(),
        label=f"{class_names[int(labels[n].get())]}"
    )

    axs[i].set_xlabel('wavelength(Å)')
    axs[i].set_title(f"{n} - {class_names[int(labels[n].get())]}")

axs[0].set_ylabel('flux')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# sklearn split still runs on CPU
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    cp.asnumpy(data),
    cp.asnumpy(labels).astype("int32"),
    test_size=0.1,
    stratify=cp.asnumpy(labels),
    shuffle=True
)

# Move back to GPU
X_train = cp.asarray(X_train)
X_test = cp.asarray(X_test)

y_train = cp.asarray(y_train)
y_test = cp.asarray(y_test)

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")

In [ ]:
# ============================================================
# GPU NORMALIZATION + GPU PCA
# ============================================================

normalizer = Normalizer(norm="l2")

X_train_norm = normalizer.fit_transform(X_train)

pca_vis = PCA(
    n_components=5,
)

X_proj = pca_vis.fit_transform(X_train_norm)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

scatter = ax.scatter(
    cp.asnumpy(X_proj[:, 0]),
    cp.asnumpy(X_proj[:, 1]),
    c=cp.asnumpy(y_train),
    s=4,
    cmap="viridis",
    alpha=0.8
)

cbar = fig.colorbar(scatter, ax=ax, ticks=[0, 1, 2])
cbar.ax.set_yticklabels(class_names)

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("GPU PCA Projection")

plt.tight_layout()
plt.show()

In [ ]:
class GPULogisticClassifier(BaseEstimator, ClassifierMixin):

    def __init__(
        self,
        C=1.0,
        pca_components=None
    ):

        self.C = C
        self.pca_components = pca_components

    def fit(self, X, y):

        X = cp.asarray(X)
        y = cp.asarray(y)

        self.normalizer = Normalizer(norm="l2")

        X = self.normalizer.fit_transform(X)

        if self.pca_components is not None:

            self.pca = PCA(
                n_components=self.pca_components,
            )

            X = self.pca.fit_transform(X)

        self.model = LogisticRegression(
            C=self.C,
            max_iter=10000
        )

        self.model.fit(X, y)

        return self

    def predict(self, X):

        X = cp.asarray(X)

        X = self.normalizer.transform(X)

        if self.pca_components is not None:
            X = self.pca.transform(X)

        pred = self.model.predict(X)

        return cp.asnumpy(pred)

In [ ]:
class GPURandomForestClassifier(BaseEstimator, ClassifierMixin):

    def __init__(
        self,
        n_estimators=200,
        max_depth=10,
        pca_components=None
    ):

        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.pca_components = pca_components

    def fit(self, X, y):

        X = cp.asarray(X)
        y = cp.asarray(y)

        self.normalizer = Normalizer(norm="l2")

        X = self.normalizer.fit_transform(X)

        if self.pca_components is not None:

            self.pca = PCA(
                n_components=self.pca_components,
            )

            X = self.pca.fit_transform(X)

        self.model = RandomForestClassifier(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
        )

        self.model.fit(X, y)

        return self

    def predict(self, X):

        X = cp.asarray(X)

        X = self.normalizer.transform(X)

        if self.pca_components is not None:
            X = self.pca.transform(X)

        pred = self.model.predict(X)

        return cp.asnumpy(pred)

In [ ]:
class GPUXGBoostClassifier(BaseEstimator, ClassifierMixin):

    def __init__(
        self,
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        pca_components=None
    ):

        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.pca_components = pca_components

    def fit(self, X, y):

        X = cp.asarray(X)
        y = cp.asarray(y)

        self.normalizer = Normalizer(norm="l2")

        X = self.normalizer.fit_transform(X)

        if self.pca_components is not None:

            self.pca = PCA(
                n_components=self.pca_components,
            )

            X = self.pca.fit_transform(X)

        self.model = xgb.XGBClassifier(

            # =================================================
            # GPU SETTINGS
            # =================================================

            device="cuda",
            tree_method="hist",

            eval_metric="mlogloss",
            objective="multi:softprob",

            n_estimators=self.n_estimators,
            learning_rate=self.learning_rate,
            max_depth=self.max_depth,
            subsample=self.subsample,
            colsample_bytree=self.colsample_bytree
        )

        self.model.fit(X, y)

        return self

    def predict(self, X):
    
        X = cp.asarray(X)
    
        X = self.normalizer.transform(X)
    
        if self.pca_components is not None:
            X = self.pca.transform(X)
    
        # =====================================================
        # KEEP DATA ON GPU
        # =====================================================
    
        pred = self.model.predict(X)
    
        # sklearn metrics expect CPU numpy arrays
        return cp.asnumpy(pred)

In [ ]:
models = {

    "logit": (
        GPULogisticClassifier(),
        {
            "C": stats.loguniform(1e-1, 1e4),
            "pca_components": [None, 10, 100, 1000]
        }
    ),

    "rf": (
        GPURandomForestClassifier(),
        {
            "n_estimators": stats.randint(100, 400),
            "max_depth": [5, 10, 20],
            "pca_components": [None, 10, 100, 1000]
        }
    ),

    "xgb": (
        GPUXGBoostClassifier(),
        {
            "n_estimators": stats.randint(100, 500),
            "learning_rate": stats.loguniform(1e-3, 1e-1),
            "max_depth": [3, 4, 5, 6],
            "subsample": [0.6, 0.8, 1.0],
            "colsample_bytree": [0.5, 0.8, 1.0],
            "pca_components": [None, 10, 100, 1000]
        }
    )
}

In [ ]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
)

In [ ]:
results = []

trained_models = {}

best_overall_model = None
best_overall_score = -np.inf
best_model_name = None

# ============================================================
# sklearn CV still expects CPU arrays
# ============================================================

X_train_cpu = cp.asnumpy(X_train)
y_train_cpu = cp.asnumpy(y_train)

X_test_cpu = cp.asnumpy(X_test)
y_test_cpu = cp.asnumpy(y_test)

for name, (model, params) in models.items():

    print(f"\n{'='*50}")
    print(f"Training {name}")
    print(f"{'='*50}")

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=params,
        n_iter=10,
        cv=cv,
        scoring="f1_macro",
        verbose=1,
        n_jobs=1,
        return_train_score=True
    )

    start = time.time()

    search.fit(X_train_cpu, y_train_cpu)

    elapsed = time.time() - start

    trained_models[name] = search

    best_model = search.best_estimator_

    pred = best_model.predict(X_test_cpu)

    acc = accuracy_score(y_test_cpu, pred)

    f1 = f1_score(
        y_test_cpu,
        pred,
        average="macro"
    )

    print("\nBest Parameters:")
    print(search.best_params_)

    print("\nBest CV Score:")
    print(f"{search.best_score_:.4f}")

    print("\nTest Metrics:")
    print(f"Accuracy : {acc:.4f}")
    print(f"F1 Macro : {f1:.4f}")

    results.append({
        "model": name,
        "accuracy": acc,
        "f1_macro": f1,
        "best_cv_score": search.best_score_,
        "fit_time_sec": elapsed,
        "best_params": search.best_params_
    })

    if search.best_score_ > best_overall_score:

        best_overall_score = search.best_score_

        best_overall_model = best_model

        best_model_name = name

In [ ]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="best_cv_score",
    ascending=False
)

print(results_df)

In [ ]:
joblib.dump(
    best_overall_model,
    "best_spectrum_classifier_gpu.pkl"
)

In [ ]:
final_model = joblib.load(
    "best_spectrum_classifier_gpu.pkl"
)

test_predictions = final_model.predict(X_test_cpu)

test_accuracy = accuracy_score(
    y_test_cpu,
    test_predictions
)

test_f1 = f1_score(
    y_test_cpu,
    test_predictions,
    average="macro"
)

print(f"Final Test Accuracy: {test_accuracy:.4f}")
print(f"Final Test F1: {test_f1:.4f}")

In [ ]:
def get_multiclass_metrics(y_actual, y_pred, classes):
    cm = confusion_matrix(y_actual, y_pred)
    
    for i, cls in enumerate(classes):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = np.sum(cm) - TP - FP - FN

        # Safe division to prevent ZeroDivisionError
        sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0

        print(f"{cls:10s} Sensitivity: {sensitivity:.4f} | Specificity: {specificity:.4f}")

def report(y_actual, y_pred, classes, title):
    print(classification_report(y_actual, y_pred, target_names=classes))
    print("\nPer-class metrics:")
    get_multiclass_metrics(y_actual, y_pred, classes)
    cm = confusion_matrix(y_actual, y_pred, normalize="true")

    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="viridis", 
                xticklabels=classes, yticklabels=classes)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.show()

In [ ]:
report(y_test.get(), test_predictions, class_names,
       title="Final Test Confusion Matrix")
